In [1]:
import torch
device = "mps" if torch.backends.mps.is_available() else "cpu"

In [2]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

/Users/harshiniramasamy/PycharmProjects/book-recommender/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from dotenv import load_dotenv

load_dotenv()

False

In [4]:
import os

print(os.listdir())

['tagged_description.txt', '.junie', '.venv', 'books_cleaned.csv', 'books_with_emotions.csv', 'data-exploration.ipynb', 'main.py', 'books_with_categories.csv', 'GRADIO.py', '.idea']


In [5]:
import pandas as pd

books = pd.read_csv("books_cleaned.csv")

In [6]:
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883A NOVEL THAT READERS and critics ...
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982A new 'Christie for Christmas' --...
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736A memorable, mesmerizing heroine ..."
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897Lewis' work on the nature of love...
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934""In The Problem of Pain, C.S. Lew..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,0.0,Mistaken Identity,9788172235222On A Train Journey Home To North ...
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,24.0,Journey to the East,9788173031014This book tells the tale of a man...
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,1568.0,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623Wisdom to Create a Life of Passio...
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535This collection of the timeless t...


In [7]:
books["tagged_description"].to_csv('tagged_description.txt', index=False,
sep="\t", header=False,)


In [8]:
from langchain_core.documents import Document

documents = []

for _, row in books.iterrows():

    documents.append(
        Document(
            page_content=row["tagged_description"],
            metadata={
                "isbn13": row["isbn13"]
            }
        )
    )

In [9]:
documents[0]

Document(metadata={'isbn13': 9780002005883}, page_content='9780002005883A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a so

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

db_books = Chroma.from_documents(documents, embedding=embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 11440.73it/s]


In [11]:
import sys
print(sys.executable)

/Users/harshiniramasamy/PycharmProjects/book-recommender/.venv/bin/python


In [12]:
import sys
!{sys.executable} -m pip install sentence-transformers langchain langchain-community langchain-chroma langchain-huggingface chromadb torch


[notice] A new release of pip is available: 26.0.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [13]:
query = "A book to teach children about education"
docs = db_books.similarity_search(query, k = 10)
docs


[Document(id='fb5e81b2-67c8-48c9-9210-e3b454ff962f', metadata={'isbn13': 9780742529786}, page_content="9780742529786This collection of Chomsky's influential writings on education builds a larger understanding of our educational needs, starting with the changing role of schools today, yet broadening our view toward new models of public education for citizenship."),
 Document(id='0d847ba3-480b-4dce-baaf-ca573c45fa14', metadata={'isbn13': 9780671631987}, page_content='9780671631987With more than half a million copies in print, Teach Your Child to Read in 100 Easy Lessons is the definitive guide to giving your child the reading skills needed now for a better chance at tomorrow, while bringing you and your child closer together. Is your child halfway through first grade and still unable to read? Is your preschooler bored with coloring and ready for reading? Do you want to help your child read, but are afraid you’ll do something wrong? Teach Your Child to Read in 100 Easy Lessons is a comple

In [14]:
import re

isbn = re.sub(r'\D', '', docs[0].page_content.split()[0])

book = books[books["isbn13"] == int(isbn)]

In [15]:
import re

def retrieve_semantic_recommendation(
        query: str,
        top_k: int = 10,
) -> pd.DataFrame:

    recs = db_books.similarity_search(query, k=50)

    books_list = []

    for i in range(len(recs)):
        books_list += [
            int(
                re.sub(
                    r'\D',
                    '',
                    recs[i].page_content.split()[0]
                )
            )
        ]

    return books[
        books["isbn13"].isin(books_list)
    ].head(top_k)

In [16]:
from langchain_community.document_loaders import TextLoader


retrieve_semantic_recommendation("a book about nature")

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description
59,9780007151240,0007151241,The Family Way,Tony Parsons,Parenthood,http://books.google.com/books/content?id=dJEIx...,It should be the most natural thing in the wor...,2005.0,3.51,400.0,2095.0,The Family Way,9780007151240It should be the most natural thi...
176,9780060594527,0060594527,Lost Horizon,James Hilton,Fiction,http://books.google.com/books/content?id=QD1s4...,Following a plane crash in the Himalayan mount...,2004.0,3.92,241.0,11794.0,Lost Horizon: A Novel,9780060594527Following a plane crash in the Hi...
215,9780060760441,0060760443,The Reading Group,Elizabeth Noble,Fiction,http://books.google.com/books/content?id=IagWj...,The Reading Group follows the trials and tribu...,2005.0,3.34,429.0,6408.0,The Reading Group: A Novel,9780060760441The Reading Group follows the tri...
254,9780060841867,0060841869,The Curtain,Milan Kundera,Literary Collections,http://books.google.com/books/content?id=JCDXP...,Traces the author's personal view of the histo...,2006.0,3.96,176.0,1351.0,The Curtain: An Essay in Seven Parts,9780060841867Traces the author's personal view...
283,9780060915438,0060915439,Holy the Firm,Annie Dillard,Literary Criticism,http://books.google.com/books/content?id=ps1Ll...,In 1975 Annie Dillard took up residence on an ...,1988.0,4.22,76.0,3562.0,Holy the Firm,9780060915438In 1975 Annie Dillard took up res...
324,9780060959036,0060959037,Prodigal Summer,Barbara Kingsolver,Fiction,http://books.google.com/books/content?id=06IwG...,Barbara Kingsolver's fifth novel is a hymn to ...,2001.0,4.00,444.0,85440.0,Prodigal Summer: A Novel,9780060959036Barbara Kingsolver's fifth novel ...
383,9780061144899,0061144894,When the Heart Waits,Sue Monk Kidd,Religion,http://books.google.com/books/content?id=JlP91...,From the Bestselling Author of The Secret Life...,2006.0,4.17,240.0,2141.0,When the Heart Waits: Spiritual Direction for ...,9780061144899From the Bestselling Author of Th...
415,9780064406307,006440630X,The Midwife's Apprentice (rpkg),Karen Cushman,Juvenile Fiction,http://books.google.com/books/content?id=Bhm76...,"'Like Cushman's 1995 Newbery Honor Book, Cathe...",1996.0,3.72,128.0,35319.0,The Midwife's Apprentice (rpkg),9780064406307'Like Cushman's 1995 Newbery Hono...
692,9780140448009,0140448004,Three Tales,Gustave Flaubert;Roger Whitehouse;Geoffrey Wall,Fiction,http://books.google.com/books/content?id=XFzga...,Features short fiction by the French naturalis...,2005.0,3.71,110.0,3050.0,Three Tales,9780140448009Features short fiction by the Fre...
937,9780156932509,0156932504,The Uses of Literature,Italo Calvino,Literary Criticism,http://books.google.com/books/content?id=INlwz...,This essay collection discusses literature in ...,1987.0,4.07,341.0,887.0,The Uses of Literature: Essays,9780156932509This essay collection discusses l...


In [17]:
category_mapping = {'Fiction' : "Fiction",
 'Juvenile Fiction': "Children's Fiction",
 'Biography & Autobiography': "Nonfiction",
 'History': "Nonfiction",
 'Literary Criticism': "Nonfiction",
 'Philosophy': "Nonfiction",
 'Religion': "Nonfiction",
 'Comics & Graphic Novels': "Fiction",
 'Drama': "Fiction",
 'Juvenile Nonfiction': "Children's Nonfiction",
 'Science': "Nonfiction",
 'Poetry': "Fiction"}

books["simple_categories"] = books["categories"].map(category_mapping)

In [18]:
books[~(books["simple_categories"].isna())]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description,simple_categories
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883A NOVEL THAT READERS and critics ...,Fiction
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736A memorable, mesmerizing heroine ...",Fiction
8,9780006482079,0006482074,Warhost of Vastmark,Janny Wurts,Fiction,http://books.google.com/books/content?id=uOL0f...,"Tricked once more by his wily half-brother, Ly...",1995.0,4.03,522.0,2966.0,Warhost of Vastmark,9780006482079Tricked once more by his wily hal...,Fiction
30,9780006646006,000664600X,Ocean Star Express,Mark Haddon;Peter Sutton,Juvenile Fiction,http://books.google.com/books/content?id=I2QZA...,Joe and his parents are enjoying a summer holi...,2002.0,3.50,32.0,1.0,Ocean Star Express,9780006646006Joe and his parents are enjoying ...,Children's Fiction
46,9780007121014,0007121016,Taken at the Flood,Agatha Christie,Fiction,http://books.google.com/books/content?id=3gWlx...,A Few Weeks After Marrying An Attractive Young...,2002.0,3.71,352.0,8852.0,Taken at the Flood,9780007121014A Few Weeks After Marrying An Att...,Fiction
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5178,9781933648279,1933648279,Night Has a Thousand Eyes,Cornell Woolrich,Fiction,http://books.google.com/books/content?id=3Gk6s...,"""Cornell Woolrich's novels define the essence ...",2007.0,3.77,344.0,680.0,Night Has a Thousand Eyes,"9781933648279""Cornell Woolrich's novels define...",Fiction
5188,9784770028969,4770028962,Coin Locker Babies,村上龍,Fiction,http://books.google.com/books/content?id=87DJw...,Rescued from the lockers in which they were le...,2002.0,3.75,393.0,5560.0,Coin Locker Babies,9784770028969Rescued from the lockers in which...,Fiction
5189,9788122200850,8122200850,"Cry, the Peacock",Anita Desai,Fiction,http://books.google.com/books/content?id=_QKwV...,This book is the story of a young girl obsesse...,1980.0,3.22,218.0,134.0,"Cry, the Peacock",9788122200850This book is the story of a young...,Fiction
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535This collection of the timeless t...,Nonfiction


In [19]:
from transformers import pipeline

fiction_categories = ["Fiction", "Nonfiction"]

pipe = pipeline("zero-shot-classification", model="facebook/bart-large-mnli", device="mps")

Loading weights: 100%|██████████| 515/515 [00:00<00:00, 7990.69it/s]


In [20]:
sequence = books.loc[books["simple_categories"] == "Fiction", "description"].reset_index(drop=True)[0]
pipe(sequence, fiction_categories)


{'sequence': 'A NOVEL THAT READERS and critics have been eagerly anticipating for over a decade, Gilead is an astonishingly imagined story of remarkable lives. John Ames is a preacher, the son of a preacher and the grandson (both maternal and paternal) of preachers. It’s 1956 in Gilead, Iowa, towards the end of the Reverend Ames’s life, and he is absorbed in recording his family’s story, a legacy for the young son he will never see grow up. Haunted by his grandfather’s presence, John tells of the rift between his grandfather and his father: the elder, an angry visionary who fought for the abolitionist cause, and his son, an ardent pacifist. He is troubled, too, by his prodigal namesake, Jack (John Ames) Boughton, his best friend’s lost son who returns to Gilead searching for forgiveness and redemption. Told in John Ames’s joyous, rambling voice that finds beauty, humour and truth in the smallest of life’s details, Gilead is a song of celebration and acceptance of the best and the worst

In [21]:
import numpy as np

predictions = pipe(sequence, fiction_categories)
max_index = np.argmax(predictions["scores"])
max_label = predictions["labels"][max_index]
max_label

'Fiction'

In [22]:
def generate_predictions(sequence,categories):
   predictions = pipe(sequence, categories)
   max_index = np.argmax(predictions["scores"])
   max_label = predictions["labels"][max_index]
   return max_label

In [23]:
from tqdm import tqdm

actual_cats =  []
predicted_cats = []

for i in tqdm(range(0, 300)):
    sequence = books.loc[books["simple_categories"] == "Fiction", "description"].reset_index(drop=True)[i]
    predicted_cats += [generate_predictions(sequence, fiction_categories)]
    actual_cats += ["Fiction"]


100%|██████████| 300/300 [00:59<00:00,  5.02it/s]


In [24]:
for i in tqdm(range(0, 300)):
    sequence = books.loc[books["simple_categories"] == "Nonfiction", "description"].reset_index(drop=True)[i]
    predicted_cats += [generate_predictions(sequence, fiction_categories)]
    actual_cats += ["Nonfiction"]

100%|██████████| 300/300 [01:23<00:00,  3.58it/s]


In [25]:
predictions_df = pd.DataFrame({"actual_categories": actual_cats, "predictions": predicted_cats})


In [26]:
predictions_df

,actual_categories,predictions
0,Fiction,Fiction
1,Fiction,Fiction
2,Fiction,Fiction
3,Fiction,Nonfiction
4,Fiction,Fiction
...,...,...
595,Nonfiction,Nonfiction
596,Nonfiction,Fiction
597,Nonfiction,Nonfiction
598,Nonfiction,Nonfiction


In [27]:
predictions_df["correct_predictions"] = predictions_df["actual_categories"] == predictions_df["predictions"]

In [28]:
predictions_df["correct_predictions"].sum()/len(predictions_df)

np.float64(0.7783333333333333)

In [29]:
isbn = []
predicted_cats = []

missing_cats = books.loc[books["simple_categories"].isna(), ["isbn13", "description"]].reset_index(drop=True)


In [30]:
for i in tqdm(range(0, len(missing_cats))):
    sequence = missing_cats["description"][i]
    predicted_cats += [generate_predictions(sequence, fiction_categories)]
    isbn += [missing_cats["isbn13"][i]]

100%|██████████| 1454/1454 [06:10<00:00,  3.92it/s] 


In [31]:
missing_predicted_df = pd.DataFrame({"isbn13": isbn, "predicted_categories": predicted_cats})

In [32]:
missing_predicted_df

,isbn13,predicted_categories
0,9780002261982,Fiction
1,9780006280897,Nonfiction
2,9780006280934,Nonfiction
3,9780006380832,Nonfiction
4,9780006470229,Fiction
...,...,...
1449,9788125026600,Nonfiction
1450,9788171565641,Fiction
1451,9788172235222,Fiction
1452,9788173031014,Nonfiction


In [33]:
books = pd.merge(books,missing_predicted_df,on="isbn13",how = "left" )
books["simple_categories"] = np.where(books["simple_categories"].isna(),books["predicted_categories"] , books["simple_categories"])
books = books.drop(columns=["predicted_categories"])

In [34]:
books

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description,simple_categories
0,9780002005883,0002005883,Gilead,Marilynne Robinson,Fiction,http://books.google.com/books/content?id=KQZCP...,A NOVEL THAT READERS and critics have been eag...,2004.0,3.85,247.0,361.0,Gilead,9780002005883A NOVEL THAT READERS and critics ...,Fiction
1,9780002261982,0002261987,Spider's Web,Charles Osborne;Agatha Christie,Detective and mystery stories,http://books.google.com/books/content?id=gA5GP...,A new 'Christie for Christmas' -- a full-lengt...,2000.0,3.83,241.0,5164.0,Spider's Web: A Novel,9780002261982A new 'Christie for Christmas' --...,Fiction
2,9780006178736,0006178731,Rage of angels,Sidney Sheldon,Fiction,http://books.google.com/books/content?id=FKo2T...,"A memorable, mesmerizing heroine Jennifer -- b...",1993.0,3.93,512.0,29532.0,Rage of angels,"9780006178736A memorable, mesmerizing heroine ...",Fiction
3,9780006280897,0006280897,The Four Loves,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=XhQ5X...,Lewis' work on the nature of love divides love...,2002.0,4.15,170.0,33684.0,The Four Loves,9780006280897Lewis' work on the nature of love...,Nonfiction
4,9780006280934,0006280935,The Problem of Pain,Clive Staples Lewis,Christian life,http://books.google.com/books/content?id=Kk-uV...,"""In The Problem of Pain, C.S. Lewis, one of th...",2002.0,4.09,176.0,37569.0,The Problem of Pain,"9780006280934""In The Problem of Pain, C.S. Lew...",Nonfiction
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
5192,9788172235222,8172235224,Mistaken Identity,Nayantara Sahgal,Indic fiction (English),http://books.google.com/books/content?id=q-tKP...,On A Train Journey Home To North India After L...,2003.0,2.93,324.0,0.0,Mistaken Identity,9788172235222On A Train Journey Home To North ...,Fiction
5193,9788173031014,8173031010,Journey to the East,Hermann Hesse,Adventure stories,http://books.google.com/books/content?id=rq6JP...,This book tells the tale of a man who goes on ...,2002.0,3.70,175.0,24.0,Journey to the East,9788173031014This book tells the tale of a man...,Nonfiction
5194,9788179921623,817992162X,The Monk Who Sold His Ferrari: A Fable About F...,Robin Sharma,Health & Fitness,http://books.google.com/books/content?id=c_7mf...,"Wisdom to Create a Life of Passion, Purpose, a...",2003.0,3.82,198.0,1568.0,The Monk Who Sold His Ferrari: A Fable About F...,9788179921623Wisdom to Create a Life of Passio...,Fiction
5195,9788185300535,8185300534,I Am that,Sri Nisargadatta Maharaj;Sudhakar S. Dikshit,Philosophy,http://books.google.com/books/content?id=Fv_JP...,This collection of the timeless teachings of o...,1999.0,4.51,531.0,104.0,I Am that: Talks with Sri Nisargadatta Maharaj,9788185300535This collection of the timeless t...,Nonfiction


In [35]:
books[books["categories"].str.lower().isin(
    [ "romance", "science-fiction", "scifi", "horror", "mystery","thriller","horror",
      "comedy","crime","historical fiction"
    ]
)]

,isbn13,isbn10,title,authors,categories,thumbnail,description,published_year,average_rating,num_pages,ratings_count,title_and_subtitle,tagged_description,simple_categories
478,9780099422341,0099422344,Yeats is Dead!,Joseph O'Connor,Comedy,http://books.google.com/books/content?id=DrE3I...,"In aid of Amnesty International, this is a bri...",2002.0,3.39,298.0,34.0,Yeats is Dead!: A Novel by Fifteen Irish Writers,"9780099422341In aid of Amnesty International, ...",Fiction
491,9780099446729,0099446723,Blackwood Farm,Anne Rice,Horror,http://books.google.com/books/content?id=cIn8T...,"Lestat Is Back, Saviour And Demon, Presiding O...",2003.0,3.86,774.0,26145.0,Blackwood Farm,"9780099446729Lestat Is Back, Saviour And Demon...",Fiction
542,9780099911708,0099911701,Cross Stitch,Diana Gabaldon,Historical fiction,http://books.google.com/books/content?id=0Pm4i...,"In 1945, Claire Randall is back from the war a...",1992.0,4.22,864.0,2319.0,Cross Stitch,"9780099911708In 1945, Claire Randall is back f...",Fiction
5031,9781857024074,1857024079,A Thousand Orange Trees,Kathryn Harrison,Historical fiction,http://books.google.com/books/content?id=TRMwG...,"Set in 17th-century Spain, this is a twisting ...",1995.0,3.61,317.0,78.0,A Thousand Orange Trees,"9781857024074Set in 17th-century Spain, this i...",Fiction


In [36]:
books.to_csv("books_with_categories.csv",index = False)

In [37]:
books.to_csv("books_with_categories.csv", index=False)
import os
print(os.path.abspath("books_with_categories.csv"))

/Users/harshiniramasamy/PycharmProjects/book-recommender/books_with_categories.csv
